# Materialise Sint lower-back acceleration + gyroscope windows

This notebook creates a **new, versioned** lower-back tensor for development transport experiments. It follows the passed Notebook 26 audit, uses the published Vicon-to-Xsens trial mapping and each trial's `sensorspec`, synchronises via shared packets, converts acceleration from m/s² to g and Xsens gyroscope values from rad/s to deg/s, and resamples only the documented 40 Hz `900_V_01` GRAIL exception.

It does not change any existing 3-channel magnitude array, model checkpoint, split, calibration, or frozen evaluation cohort.

In [1]:
import json
from fractions import Fraction
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import resample_poly

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW = PROJECT_ROOT / 'data/raw/sint_maartenskliniek/extracted/IMU_GaitAnalysis-1.1.0/data'
PROCESSED = PROJECT_ROOT / 'data/processed'
INTERIM = PROJECT_ROOT / 'data/interim'
MANIFEST = pd.read_csv(PROCESSED / 'sint_maartenskliniek_trial_manifest.csv')
MAPPED = MANIFEST.loc[MANIFEST.export_exists].copy().sort_values(['participant', 'exported_trial'])
TARGET_FS, WINDOW_SAMPLES, HOP_SAMPLES = 100, 500, 250
G, RAD_TO_DEG = 9.80665, 180.0 / np.pi
REQUIRED = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyr_X', 'Gyr_Y', 'Gyr_Z']
print(f'Materialising {len(MAPPED)} mapped trials from {MAPPED.participant.nunique()} participants.')

Materialising 79 mapped trials from 30 participants.


In [2]:
def read_export(path):
    lines = path.read_text(errors='replace').splitlines()
    header = next(i for i, line in enumerate(lines) if line.startswith('PacketCounter\t'))
    return pd.read_csv(path, sep='\t', skiprows=header, low_memory=False)

def participant_folder(label):
    return RAW / ('CVA' if label == 'stroke' else 'Healthy_controls')

def load_packet_aligned_lower_back(record):
    trial_dir = participant_folder(record.label) / record.participant / 'Xsens' / record.exported_trial
    spec = json.loads((trial_dir / 'sensorspec.json.txt').read_text(encoding='utf-8'))
    frames = {}
    for location in ('lumbar', 'leftfoot', 'rightfoot'):
        matches = list(trial_dir.glob(f"*_{spec[location]}"))
        assert len(matches) == 1, (record.participant, record.exported_trial, location, matches)
        frame = read_export(matches[0])
        assert set(REQUIRED + ['PacketCounter']).issubset(frame.columns)
        frame = frame[['PacketCounter'] + REQUIRED].copy()
        frame = frame.apply(pd.to_numeric, errors='coerce').dropna()
        frame = frame.drop_duplicates('PacketCounter').set_index('PacketCounter')
        frames[location] = frame
    common = frames['lumbar'].index.intersection(frames['leftfoot'].index).intersection(frames['rightfoot'].index).sort_values()
    assert len(common) >= WINDOW_SAMPLES
    lb = frames['lumbar'].loc[common, REQUIRED].to_numpy(np.float64)
    lb[:, :3] /= G
    lb[:, 3:] *= RAD_TO_DEG
    source_fs = 40 if (record.participant == '900_V_01' and record.trial_type != '2MWT') else 100
    if source_fs != TARGET_FS:
        ratio = Fraction(TARGET_FS / source_fs).limit_denominator(1000)
        lb = resample_poly(lb, ratio.numerator, ratio.denominator, axis=0)
    assert np.isfinite(lb).all()
    return lb.astype(np.float32), int(len(common)), source_fs

# A lower-back magnitude representation is orientation-robust and preserves the original RQ placement.
# The two channels retain complementary trunk acceleration and angular-velocity dynamics.
def lower_back_magnitudes(lb_six_dof):
    return np.column_stack([
        np.linalg.norm(lb_six_dof[:, :3], axis=1),
        np.linalg.norm(lb_six_dof[:, 3:], axis=1),
    ]).astype(np.float32)

In [3]:
windows, metadata_rows, trial_rows = [], [], []
for record in MAPPED.itertuples(index=False):
    try:
        lb, common_packets, source_fs = load_packet_aligned_lower_back(record)
        signal = lower_back_magnitudes(lb)
        starts = list(range(0, len(signal) - WINDOW_SAMPLES + 1, HOP_SAMPLES))
        for start in starts:
            windows.append(signal[start:start + WINDOW_SAMPLES])
            metadata_rows.append({
                'window_id': len(metadata_rows), 'dataset_id': 'sint_maartenskliniek',
                'participant_key': record.participant_key, 'participant': record.participant,
                'trial': record.exported_trial, 'label': record.label, 'label_binary': record.label_binary,
                'source_fs_hz': source_fs, 'window_fs_hz': TARGET_FS, 'window_seconds': WINDOW_SAMPLES / TARGET_FS,
                'hop_seconds': HOP_SAMPLES / TARGET_FS, 'vicon_trial': record.vicon_trial,
                'trial_type': record.trial_type, 'speed_condition': record.speed_condition,
                'selection_rule': 'published_vicon_xsens_mapping_packet_intersection_all_valid_windows',
                'adapter_version': 'lb_acc_gyr_v1_2026-09-02',
            })
        trial_rows.append({'participant': record.participant, 'trial': record.exported_trial, 'label': record.label,
                           'source_fs_hz': source_fs, 'common_packets_before_resampling': common_packets,
                           'samples_at_100hz': len(signal), 'windows': len(starts), 'status': 'included'})
    except Exception as exc:
        trial_rows.append({'participant': record.participant, 'trial': record.exported_trial, 'label': record.label,
                           'status': f'error:{type(exc).__name__}', 'error': str(exc)})

trial_audit = pd.DataFrame(trial_rows)
assert trial_audit.status.eq('included').all(), trial_audit.loc[~trial_audit.status.eq('included')]
array = np.asarray(windows, dtype=np.float32)
metadata = pd.DataFrame(metadata_rows)
assert array.ndim == 3 and array.shape[1:] == (WINDOW_SAMPLES, 2), array.shape
assert len(array) == len(metadata) and np.isfinite(array).all()
display(trial_audit.groupby(['label', 'source_fs_hz']).agg(trials=('trial', 'size'), windows=('windows', 'sum'), participants=('participant', 'nunique')))
display(metadata.groupby('label').agg(participants=('participant', 'nunique'), windows=('window_id', 'size')))

trials  windows  participants
label   source_fs_hz                               
healthy 40                 2       93             1
        100               57     2976            20
stroke  100               20      984            10

,participants,windows
label,,
healthy,20,3069
stroke,10,984


In [4]:
array_path = PROCESSED / 'sint_lower_back_accel_gyro_windows_v1_float32.npy'
metadata_path = PROCESSED / 'sint_lower_back_accel_gyro_window_metadata_v1.csv'
trial_audit_path = INTERIM / 'sint_lower_back_accel_gyro_materialization_audit_v1.csv'
np.save(array_path, array)
metadata.to_csv(metadata_path, index=False)
trial_audit.to_csv(trial_audit_path, index=False)
print(f'Array: {array_path.relative_to(PROJECT_ROOT)} | shape={array.shape} | {array_path.stat().st_size / 1024**2:.1f} MB')
print(f'Metadata: {metadata_path.relative_to(PROJECT_ROOT)}')
print(f'Trial audit: {trial_audit_path.relative_to(PROJECT_ROOT)}')
print('No existing magnitude tensor, checkpoint, or frozen cohort was modified.')

Array: data\processed\sint_lower_back_accel_gyro_windows_v1_float32.npy | shape=(4053, 500, 2) | 15.5 MB
Metadata: data\processed\sint_lower_back_accel_gyro_window_metadata_v1.csv
Trial audit: data\interim\sint_lower_back_accel_gyro_materialization_audit_v1.csv
No existing magnitude tensor, checkpoint, or frozen cohort was modified.


## Next controlled experiment

Compare lower-back acceleration-only against acceleration-plus-gyroscope using Felius, Voisard, and this versioned Sint tensor. Fit standardisation within each training fold only, use participant-disjoint source-transport evaluation, and leave all frozen cohorts untouched.